In [1]:
!pip3 install tabulate

In [3]:
# Import the required json results
import json
import os

files = {
    "TPC-C" : ["tpcc/tpcc_mono.json", "tpcc/tpcc_best.json", "tpcc/tpcc_full.json"],
    "findmates": ["find_sports/find_sport_mates_mono.json", "find_sports/find_sport_mates_best_full.json", "find_sports/find_sport_mates_best_full.json"],
    "jpabook": ["jpabook/jpabook_mono.json", "jpabook/jpabook_best.json", "jpabook/jpabook_full.json"],
    "jpetstore": ["jpetstore/jpetstore_mono.json", "jpetstore/jpetstore_best.json", "jpetstore/jpetstore_full.json" ],
    "petclinic": ["petclinic/petclinic_mono.json", "petclinic/petclinic_best.json", "petclinic/petclinic_full.json"],
    "myweb": ["myweb/myweb_mono.json", "myweb/myweb_best.json", "myweb/myweb_full.json"],
    "react": ["react/react_mono.json", "react/react_best.json", "react/react_full.json"]
}

results = {
    "TPC-C" : [],
    "findmates": [],
    "jpabook": [],
    "jpetstore": [],
    "petclinic": [],
    "myweb": [],
    "react": []
}

anomaly_labeling = {
    'Dirty Reads' : 'DR',
    'Dirty Writes' : 'DW',
    'Lost Updates/Write Skews' : 'LU/WS',
    'Lost Updates' : 'LU',
    'Non-Repeatable Reads' : 'NRR',
    'Phantom Reads' : 'PR',
    'Read Skews' : 'RS',
    'Extensions' : 'Ext'
}

results_path = "results/json"
dc_results_path = f"{results_path}/dc"

for benchmark in files:
    for file in files[benchmark]:
        path = f"{dc_results_path}/{file}"
        if os.path.exists(path):
            with open(path, "r") as f:
                results[benchmark] += [json.load(f)]
        else:
            results[benchmark] += [{}]

assert os.path.exists(f"{dc_results_path}/tpcc/tpcc_full.json"), f"TPCC Full experiment (with path {dc_results_path}/tpcc/tpcc_full.json) does not exist but is required"
    
with open(f"{dc_results_path}/tpcc/tpcc_full.json", "r") as f:
    tpcc_full_data = json.load(f)


In [4]:
#Table 6 - TPC-C full core anomalies entities
from tabulate import tabulate

entities = tpcc_full_data['perEntiteInstances']

entity_table = []

for entity_set in entities:
    anomalies = [anomaly_labeling[anomaly] for anomaly in entities[entity_set]['anomalyTypes']]
    entity_table.append([entity_set, entities[entity_set]['occurences'], anomalies])

entity_table.sort(key= lambda x: (x[1], x[0]), reverse=True)


# Creating a table with headers and a grid format
table = tabulate(
    entity_table, 
    headers=["Entities", "#Anomalies", "Anomalies Types"], 
    tablefmt="grid"
)

print(table)

+-------------------------+--------------+-------------------+
| Entities                |   #Anomalies | Anomalies Types   |
+=========================+==============+===================+
| [oorder, order_line]    |            5 | ['DW', 'RS']      |
+-------------------------+--------------+-------------------+
| [new_order, order_line] |            4 | ['RS', 'LU/WS']   |
+-------------------------+--------------+-------------------+
| [customer, warehouse]   |            4 | ['DW']            |
+-------------------------+--------------+-------------------+
| [customer, district]    |            4 | ['DW']            |
+-------------------------+--------------+-------------------+
| [new_order, oorder]     |            3 | ['RS', 'LU/WS']   |
+-------------------------+--------------+-------------------+
| [customer, new_order]   |            2 | ['RS', 'LU/WS']   |
+-------------------------+--------------+-------------------+
| [order_line, stock]     |            1 | ['RS']      

In [7]:
#Table 7 - TPC-C full core anomalies sub-transactions

sub_transactions = tpcc_full_data['perSubTransactionAnomaly']

sub_transactions_table = []

for sub_transaction_set in sub_transactions:
    functionalities = sub_transactions[sub_transaction_set]['functionalityAnomalies']
    anomaly_occurences = sub_transactions[sub_transaction_set]['anomalies']['occurences']
    anomalies = [anomaly_labeling[anomaly] for anomaly in sub_transactions[sub_transaction_set]['anomalies']['anomalyTypes']]
    
    sub_transactions_table.append([functionalities, sub_transaction_set, anomaly_occurences, anomalies])

sub_transactions_table.sort(key= lambda x: (x[2], x[0], x[1], x[3]), reverse=True)


# Creating a table with headers and a grid format
func_table = tabulate(
    sub_transactions_table, 
    headers=["Functionalities", "Sub-transactions", "#Anomalies", "Anomalies Types"], 
    tablefmt="grid"
)

print(func_table)

+-------------------------+--------------------------------------------------------+--------------+-------------------+
| Functionalities         | Sub-transactions                                       |   #Anomalies | Anomalies Types   |
+=========================+========================================================+==============+===================+
| [payment]               | [payment_1, payment_2]                                 |            4 | ['DW']            |
+-------------------------+--------------------------------------------------------+--------------+-------------------+
| [payment]               | [payment_0, payment_2]                                 |            4 | ['DW']            |
+-------------------------+--------------------------------------------------------+--------------+-------------------+
| [newOrder, orderStatus] | [newOrder_3, newOrder_7, orderStatus_1, orderStatus_2] |            2 | ['RS']            |
+-------------------------+-------------

In [10]:
# Table 4 - Anomalies Detected

anomalies_detected_data = []

for benchmark in results:

    if len(results[benchmark]) == 0:
        continue

    has_values = True
    for result in results[benchmark]:
        if len(result) == 0:
            has_values = False
            break

    if not has_values:
        print(f"Benchmark {benchmark} is missing decompositions. Skippping")
        continue
    mono = results[benchmark][0]
    best = results[benchmark][1]
    full = results[benchmark][2]

    anomaly_detected_row = [benchmark]

    anomaly_detected_row += [f"{mono['coreAnomalies']} | {best['coreAnomalies']} | {full['coreAnomalies']}"]
    anomaly_detected_row += [f"{mono['totalAnomalies']} | {best['totalAnomalies']} | {full['totalAnomalies']}"]
    anomaly_detected_row += [f"{mono['executionTime'] / 1000} | {best['executionTime'] / 1000} | {full['executionTime'] / 1000}"]

    anomalies_detected_data.append(anomaly_detected_row)


per_type_table = tabulate(
    anomalies_detected_data, 
    headers=["Benchmark", "Core Anomalies\nmono|'best'|full", "Total Anomalies\nmono|'best'|full", "Execution Time (s)\nmono|'best'|full"], 
    tablefmt="grid"
)

print(per_type_table)

Benchmark findmates is missing decompositions. Skippping
Benchmark jpabook is missing decompositions. Skippping
Benchmark jpetstore is missing decompositions. Skippping
Benchmark petclinic is missing decompositions. Skippping
Benchmark myweb is missing decompositions. Skippping
Benchmark react is missing decompositions. Skippping
+-------------+--------------------+--------------------+--------------------------+
| Benchmark   | Core Anomalies     | Total Anomalies    | Execution Time (s)       |
|             | mono|'best'|full   | mono|'best'|full   | mono|'best'|full         |
+=============+====================+====================+==========================+
| TPC-C       | 0 | 0 | 28         | 0 | 0 | 98         | 6.005 | 10.215 | 528.087 |
+-------------+--------------------+--------------------+--------------------------+


In [11]:
# Table 5 - Mad anomalies found per type:
per_type_data = []


def getAnomalies(exp):
    anomalies = exp["anomalyInstances"]

    occurences = []
    
    for anomaly in anomaly_labeling:
        if anomaly in anomalies:
            occurences += [str(anomalies[anomaly])]
        else: 
            occurences += ["0"]

    occurences += [str(exp['coreAnomalies'])]

    return occurences
    
for benchmark in results:

    if len(results[benchmark]) == 0:
        print(f"Benchmark {benchmark} does not have any experiments. Skippping")
        continue

    if len(results[benchmark][1]) == 0:
        print(f"Benchmark {benchmark} is missing 'Best' decomposition. Skippping")
        continue
    
    if len(results[benchmark][2]) == 0:
        print(f"Benchmark {benchmark} is missing Full decomposition. Skippping")
        continue
        
    best = getAnomalies(results[benchmark][1])
    full = getAnomalies(results[benchmark][2])

    benchmark_anomaly_row = [benchmark]
    
    for i in range(len(best)):
        benchmark_anomaly_row += [f"{best[i]} / {full[i]}"]
    
    per_type_data.append(benchmark_anomaly_row)
    
per_type_table = tabulate(
    per_type_data, 
    headers=["Benchmark", "#DR\n'best'/full", "#DW\n'best'/full", "#LU/WS\n'best'/full", "#LU\n'best'/full", "#NRR\n'best'/full", "#PR\n'best'/full", "#RS\n'best'/full", "#Ext\n'best'/full", "#Total\n'best'/full"], 
    tablefmt="grid"
)

print(per_type_table)


Benchmark findmates is missing 'Best' decomposition. Skippping
Benchmark jpabook is missing 'Best' decomposition. Skippping
Benchmark jpetstore is missing 'Best' decomposition. Skippping
Benchmark petclinic is missing 'Best' decomposition. Skippping
Benchmark myweb is missing 'Best' decomposition. Skippping
Benchmark react is missing 'Best' decomposition. Skippping
+-------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+
| Benchmark   | #DR           | #DW           | #LU/WS        | #LU           | #NRR          | #PR           | #RS           | #Ext          | #Total        |
|             | 'best'/full   | 'best'/full   | 'best'/full   | 'best'/full   | 'best'/full   | 'best'/full   | 'best'/full   | 'best'/full   | 'best'/full   |
+=============+===============+===============+===============+===============+===============+===============+===============+===============+=========

In [32]:
# Table 8 - Divide and Conquer Performance (s)

search_technique_data = []

search_technique_table = tabulate(
    search_technique_data, 
    headers=["Benchmark", "Without\nmono|'best'|full", "Sequential\nmono|'best'|full", "Parallel\nmono|'best'|full"], 
    tablefmt="grid"
)

print(search_technique_table)

+-------------+--------------------+--------------------+--------------------+
| Benchmark   | Without            | Sequential         | Parallel           |
|             | mono|'best'|full   | mono|'best'|full   | mono|'best'|full   |
+=============+====================+====================+====================+
+-------------+--------------------+--------------------+--------------------+
